In [11]:
import random
import time
import numpy as np
from tqdm import tqdm

In [ ]:
class RSA:
    """
    RSA implementation for encryption, signature and station-to-station key exchange.
    """

    def __init__(self, private_key: int = None, public_key: int = None, n: int = None, fast_exp: callable = None) -> None:
        self.private_key = private_key
        self.public_key = public_key
        self.n = n
        self.challenge = "Bravo ! Je suis épousplouffé par ta maîtrise du timing attack sur RSA !"
        self.fast_exp = fast_exp if fast_exp is not None else self._default_fast_exp
    
    def copy(self):
        return RSA(self.private_key, self.public_key, self.n, self.fast_exp)

    def _default_fast_exp(self, y: int, x: int, n: int) -> int:
        """Apply fast exponentiation for y^x modulo n"""
        s = 1
        y %= n

        while x > 0:
            if (x % 2) == 1:
                s = (s * y) % n
            y = (y * y) % n
            x = x >> 1

        return s

    def setKeys(self, private_key: int, public_key: int, n: int) -> None:
        self.private_key = private_key
        self.public_key = public_key
        self.n = n

    def getKeys(self) -> list:
        return [self.public_key, self.n], [self.private_key, self.n]
    
    def exportPublicKey(self) -> list:
        return [self.public_key, self.n]

    def createKeyPair(self, size: int) -> list:
        p = self.primary_nb_generator(2**(size//2 - 1), 2**(size//2))
        q = self.primary_nb_generator(2**(size//2 - 1), 2**(size//2))
        n, e, d = self.key_generator(p, q, size=size)
        self.setKeys(d, e, n)
        return [n, e, d]

    def encrypt(self, message: int) -> int:
        return self.fast_exp(message, self.public_key, self.n)

    def decrypt(self, cipher: int, private_key: int = None) -> int:
        if private_key is not None:
            return self.fast_exp(cipher, private_key, self.n)
        return self.fast_exp(cipher, self.private_key, self.n)
    
    def createChallenge(self) -> int:
        challenge_int = int.from_bytes(self.challenge.encode(), 'big')
        self.challenge = self.encrypt(challenge_int)
        return self.challenge
    
    def decryptChallenge(self, tested_key: int) -> str:
        decrypted_challenge_int = self.decrypt(self.challenge, private_key=tested_key)
        decrypted_challenge_bytes = decrypted_challenge_int.to_bytes((decrypted_challenge_int.bit_length() + 7) // 8, 'big')
        return decrypted_challenge_bytes.decode()

    def fermat_test(self, n: int) -> bool:
        """
        Check if a number is primary by running Fermat test.
        """
        for _ in range(20):
            alpha = random.randint(2, n - 1)
            if self.fast_exp(alpha, n - 1, n) != 1:
                return False
        return True

    def primary_nb_generator(self, a: int, b: int, safe_prime: bool = False) -> int:
        """
        Generate primary number in range a, b

        Parameters
        ----------
        a : int
            lower bound
        b : int
            upper bound
        safe_prime : bool
            Generate primary number p with (p - 1) / 2 also primary

        Returns
        -------
        int
            Primary number generated

        """
        while True:
            p = random.randint(a, b)
            if self.fermat_test(p):
                if safe_prime:
                    if self.fermat_test((p - 1) / 2):
                        return p
                else:
                    return p

    def Euclide(self, a: int, b: int) -> list:
        """
        Euclide's extended algorithm, used to find decryption exponent d for RSA
        Return a list [pgcd(a, b), inverse of a mod b, inverse of b mod a]

        """
        if b > a:
            a, b = b, a  # swap

        r_0, r_1 = a, b
        s_0, s_1 = 1, 0
        t_0, t_1 = 0, 1

        while r_1 != 0:
            q = r_0 // r_1
            r_0, r_1 = r_1, r_0 - q * r_1
            s_0, s_1 = s_1, s_0 - q * s_1
            t_0, t_1 = t_1, t_0 - q * t_1

        return [r_0, s_0 % b, t_0 % a]

    def key_generator(self, p: int = 0, q: int = 0, e: int = 0, size: int = 512) -> list:
        """
        Generate public and private key for RSA
        """
        if p == 0:
            p = self.primary_nb_generator(2**(size//2 - 1), 2**(size//2))
        if q == 0:
            q = self.primary_nb_generator(2**(size//2 - 1), 2**(size//2))

        assert self.fermat_test(p), "p is not primary"
        assert self.fermat_test(q), "q is not primary"
        n = p * q
        phi_n = (p - 1) * (q - 1)

        if e != 0:
            pgcd, _, d = self.Euclide(phi_n, e)
            assert pgcd == 1, "pgcd(phi_n, e) is not equal to 1, so no private key found !"

        else:
            # Find a primary number e with phi_n
            while True:
                e = random.randint(2**(size - 1), 2**(size))
                pgcd, _, d = self.Euclide(phi_n, e)

                # If primary number with phi_n, claim public key
                if pgcd == 1:
                    break
        return n, e, d

In [8]:
#----------------------------------------------------------------------------
# Examples values
#----------------------------------------------------------------------------

# for RSA encryption :

p_A = 13109499994810966779468866046493465498469807493634236479294124421385342920350717814807375283698575766763256101470694189234369358996750113963585617491399169
q_A = 9497561827984502554523100157901534504433126034087863778629488755692649311435921364240405549590851856701860175924335776598684751639633322074428628372725777
n_A = p_A * q_A
e_A = 4574830074548708213
m_1 = 123456789132456789

# Expected results :
# 1) Check if p and q are indeed prime, then compute d_A and the ciphertext.
# 2) Expected values for d_A and the ciphertext (d_A is also used for station-to-station later)
d_A = 1685394382767324790326942621450485552187209875614438478305225564629345944620726038114923060947436330701451901921041511234432041036987266468290187679773130363479895993621867708066144608084390089775045890165825736468637468786667820591136139480545376198614216373031208691260339805721685482401743494212035728605
# cipher = 32468932964181322647810913060097066975304467072050643211304428656476623133068329653886195740426516038144100129255895281039142864272296799126753030014464755203797098445143314298922512718785433009136404533290100525054356166805463645892708927694801117827432767298393815743170470207262077229267156532545837844746
# 3) And show the decryption indeed finds the correct message m_1.

print("\n=============== RSA ENCRYPTION ===============\n")

RSA_instance = RSA()
n, e, d = RSA_instance.key_generator(p_A, q_A, e_A)
RSA_instance.setKeys(d, e, n)
assert d == d_A, "Private key generated is different from the expected key !"

print(f"Message:                {m_1}\n")

cipher = RSA_instance.encrypt(m_1)
assert cipher == 32468932964181322647810913060097066975304467072050643211304428656476623133068329653886195740426516038144100129255895281039142864272296799126753030014464755203797098445143314298922512718785433009136404533290100525054356166805463645892708927694801117827432767298393815743170470207262077229267156532545837844746, "Encrypted message is not correct !"
print(f"Ciphertext:             {cipher}\n")

recovered_message = RSA_instance.decrypt(cipher)
assert recovered_message == m_1, "Recovered message is different from original message !"
print(f"Decrypted ciphertext:   {recovered_message}")


=============== RSA ENCRYPTION ===============

Message:                123456789132456789

Ciphertext:             32468932964181322647810913060097066975304467072050643211304428656476623133068329653886195740426516038144100129255895281039142864272296799126753030014464755203797098445143314298922512718785433009136404533290100525054356166805463645892708927694801117827432767298393815743170470207262077229267156532545837844746

Decrypted ciphertext:   123456789132456789


In [30]:
def collect_samples(RSA_instance: RSA, num_samples: int = 1000, num_repetitions: int = 1, server: bool = True, private_key: int = None) -> list:
    """
    Collect samples of decryption times for different cipher texts.

    Parameters
    ----------
    RSA_instance : RSA
        An instance of the RSA class.
    num_samples : int
        Number of samples to collect.
    num_repetitions : int
        Number of times to repeat each measurement.
    server : bool
        Whether to simulate the server side.
    private_key : int
        The private key to test (only used when server=False).

    Returns
    -------
    list
        A list of tuples containing (tested_cipher, decryption_time).
    """
    samples = []

    for _ in tqdm(range(num_samples), desc="Collecting server samples" if server else "Collecting samples"):
        # Generate a random cipher text to test
        tested_cipher = random.randint(1, RSA_instance.n - 1)

        decryption_times = []

        # In case of server
        if server:
            for _ in range(num_repetitions):
                start_time = time.perf_counter_ns()
                RSA_instance.decrypt(tested_cipher)
                end_time = time.perf_counter_ns()
                decryption_times.append(end_time - start_time)
            decryption_time = np.mean(decryption_times)

        # In case of client
        else:
            assert private_key is not None, "Private key must be provided when server=False"
            start_time = time.perf_counter_ns()
            for _ in range(num_repetitions):
                RSA_instance.decrypt(tested_cipher, private_key=private_key)
            end_time = time.perf_counter_ns()
            decryption_time = end_time - start_time
            decryption_time /= num_repetitions
        
        samples.append([tested_cipher, decryption_time])

    return np.array(samples)

In [46]:
def timing_attack(RSA_instance: RSA, num_samples: int = 1000, num_repetitions: int = 1, num_iterations: int = 10, num_known_bits: int = 5, buffer_size: int = 5) -> int:
    """
    Perform a timing attack on the RSA instance to recover the private key.

    Parameters
    ----------
    RSA_instance : RSA
        An instance of the RSA class.
    num_samples : int
        Number of samples to collect for the attack.
    num_repetitions : int
        Number of times to repeat each measurement.
    num_iterations : int
        Number of iterations to perform.
    num_known_bits : int
        Number of known bits of the private key.
    buffer_size : int
        The size of the buffer for the beam search.

    Returns
    -------
    int
        The recovered private key.
    """
    server_samples = collect_samples(RSA_instance, num_samples, num_repetitions, server=True)
    initial_key = RSA_instance.private_key & ((1 << num_known_bits) - 1)

    # buffer of candidate keys
    candidate_keys = np.full(buffer_size, initial_key)

    # variance associated with each key
    candidate_variances = np.full(buffer_size, np.inf)

    # historical data
    history_keys = np.zeros((num_iterations, buffer_size))
    history_variances = np.zeros((num_iterations, buffer_size))

    error_rate = 0

    for i in range(num_iterations):

        tested_keys = []
        tested_variances = []

        # We test 2 hypotheses for each key in the buffer
        for key in candidate_keys:

            key_h0 = key
            key_h1 = key | (1 << (num_known_bits + i))

            h0_samples = collect_samples(RSA_instance, num_samples, num_repetitions, server=False, private_key=key_h0)
            h0_variance = np.var(server_samples[:, 1] - h0_samples[:, 1])

            tested_keys.append(key_h0)
            tested_variances.append(h0_variance)

            h1_samples = collect_samples(RSA_instance, num_samples, num_repetitions, server=False, private_key=key_h1)
            h1_variance = np.var(server_samples[:, 1] - h1_samples[:, 1])

            tested_keys.append(key_h1)
            tested_variances.append(h1_variance)

        tested_keys = np.array(tested_keys)
        tested_variances = np.array(tested_variances)

        # We keep the best buffer_size hypotheses
        best_indices = np.argsort(tested_variances)[:buffer_size]

        candidate_keys = tested_keys[best_indices]
        candidate_variances = tested_variances[best_indices]

        # Update historical data
        history_keys[i] = candidate_keys
        history_variances[i] = candidate_variances

        # debug : best key in the buffer and its associated bit
        best_key = candidate_keys[0]
        bit_guessed = (best_key >> (num_known_bits + i)) & 1
        bit_private_key = (RSA_instance.private_key >> (num_known_bits + i)) & 1

        print(f"Iteration {i + 1}/{num_iterations}: bit guessed {bit_guessed} (h0 variance = {h0_variance}, h1 variance = {h1_variance}) {'✓ CORRECT' if bit_guessed == bit_private_key else '✗ WRONG'}")

    # Compute error rate for each candidate key in the buffer
    error_rate = []
    for key, variance in zip(candidate_keys, candidate_variances):
        guessed_key = (key >> num_known_bits) % (1 << num_iterations)
        real_key = (RSA_instance.private_key >> num_known_bits) % (1 << num_iterations)
        errors = (guessed_key ^ real_key).bit_count()
        error_rate.append(errors / num_iterations * 100)
        print(f"Candidate key: {key}, Variance: {variance}, Error rate: {error_rate[-1]:.2f}%")
    return candidate_keys, error_rate

In [47]:
timing_attack(RSA_instance, num_samples=100, num_repetitions=10, num_iterations=10, num_known_bits=5, buffer_size=1)

Iteration 1/10: bit guessed 1 (h0 variance = 4343140838.189999, h1 variance = 4332407345.389995) ✗ WRONG


Iteration 2/10: bit guessed 1 (h0 variance = 4414674000.959999, h1 variance = 4314388914.75) ✓ CORRECT


Iteration 3/10: bit guessed 0 (h0 variance = 4320559100.0, h1 variance = 4339644371.759997) ✗ WRONG


Iteration 4/10: bit guessed 1 (h0 variance = 4346571667.240004, h1 variance = 4331167028.189999) ✗ WRONG


Iteration 5/10: bit guessed 1 (h0 variance = 4428605373.790005, h1 variance = 4348616441.959997) ✗ WRONG


...

Iteration 7/10: bit guessed 0 (h0 variance = 4333366671.0, h1 variance = 4335607439.040005) ✓ CORRECT


Iteration 8/10: bit guessed 1 (h0 variance = 4382717223.959998, h1 variance = 4348284830.0) ✓ CORRECT


Iteration 9/10: bit guessed 1 (h0 variance = 4341630060.839995, h1 variance = 4317788508.990005) ✗ WRONG


Iteration 10/10: bit guessed 0 (h0 variance = 4301972259.56, h1 variance = 4341025796.0) ✗ WRONG
Candidate key: 13181, Variance: 4301972259.56, Error rate: 70.00%


(array([13181]), [70.0])